<a href="https://colab.research.google.com/github/salavii/SOP-Generator-Fine-tuning/blob/main/notebooks/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation for SOP Fine-tuning

This notebook prepares the Statement of Purpose (SOP) dataset for fine-tuning:
- Loads 500+ SOPs from multiple sources
- Formats data for instruction fine-tuning
- Splits into training and validation sets
- Exports to JSONL format

**Dataset composition:**
- 60 real-world SOPs
- 300 augmented variations
- 140 synthetic SOPs

In [2]:
!pip install -q datasets transformers

## **Upload Data**

In [3]:
# Upload dataset files
from google.colab import files
import zipfile
import os

print("Upload your 3 zip files:")
print("1. SOP_Dataset.zip (60 original)")
print("2. augmented_sops.zip (300 augmented)")
print("3. synthetic_sops.zip (140 synthetic)\n")

uploaded = files.upload()

# Extract all uploaded files
for filename in uploaded.keys():
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        folder_name = filename.replace('.zip', '')
        zip_ref.extractall(folder_name)
    print(f"✓ Extracted to {folder_name}/")

print("\n✓ All files uploaded and extracted!")

Upload your 3 zip files:
1. SOP_Dataset.zip (60 original)
2. augmented_sops.zip (300 augmented)
3. synthetic_sops.zip (140 synthetic)



Saving augmented_sops.zip to augmented_sops.zip
Saving SOP_Dataset.zip to SOP_Dataset.zip
Saving synthetic_sops.zip to synthetic_sops.zip
Extracting augmented_sops.zip...
✓ Extracted to augmented_sops/
Extracting SOP_Dataset.zip...
✓ Extracted to SOP_Dataset/
Extracting synthetic_sops.zip...
✓ Extracted to synthetic_sops/

✓ All files uploaded and extracted!


## Load and Parse SOPs

We need to extract two key pieces from each file:
1. ***Field*** (CS, Biology, etc.) - for instruction prompts
2. ***SOP text*** - the actual content for training

Files may have metadata separated by `---` or be plain text.

In [4]:
# Load all SOP files
from pathlib import Path
from typing import List, Dict, Tuple

def load_sop_file(file_path: Path) -> Dict[str, str]:
    """
    Load and parse a single SOP file.

    Args:
        file_path: Path to SOP text file

    Returns:
        Dictionary with 'metadata', 'field', and 'text' keys
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Split metadata and SOP text
    if '---' in content:
        parts = content.split('---', 1)
        metadata = parts[0].strip()
        sop_text = parts[1].strip()
    else:
        metadata = ""
        sop_text = content

    # Extract field from metadata
    field = "Computer Science"  # default
    for line in metadata.split('\n'):
        if 'Field:' in line:
            field = line.split('Field:')[1].strip()
            break

    return {
        'metadata': metadata,
        'field': field,
        'text': sop_text,
        'filename': file_path.name
    }


def load_all_sops(folders: List[str]) -> List[Dict]:
    """
    Load SOPs from multiple folders.

    Args:
        folders: List of folder paths containing SOP files

    Returns:
        List of parsed SOP dictionaries
    """
    all_sops = []

    for folder in folders:
        if not Path(folder).exists():
            print(f"⚠ Folder not found: {folder}")
            continue

        sop_files = list(Path(folder).glob('**/*.txt'))
        print(f"Loading {len(sop_files)} files from {folder}...")

        for file_path in sop_files:
            try:
                sop_data = load_sop_file(file_path)
                all_sops.append(sop_data)
            except Exception as e:
                print(f"✗ Error loading {file_path.name}: {e}")

    return all_sops


# Load all SOPs
folders = ['SOP_Dataset', 'augmented_sops', 'synthetic_sops']
dataset = load_all_sops(folders)

print(f"\n{'='*50}")
print(f"✓ Loaded {len(dataset)} SOPs total")
print(f"{'='*50}")

Loading 60 files from SOP_Dataset...
Loading 300 files from augmented_sops...
Loading 140 files from synthetic_sops...

✓ Loaded 500 SOPs total


## Format Data for Instruction Fine-tuning

Convert SOPs to chat format that Llama expects:
- **System message**: Define the assistant's role
- **User message**: The instruction (e.g., "Write SOP for CS")
- **Assistant message**: The actual SOP text

This format teaches the model input→output mapping.

In [12]:
import json
# Format data for instruction fine-tuning
def format_for_training(sop_data: Dict[str, str]) -> Dict:
    """
    Convert SOP to instruction fine-tuning format.

    Args:
        sop_data: Dictionary with 'field' and 'text' keys

    Returns:
        Dictionary in chat format for training
    """
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert at writing compelling Statements of Purpose for graduate school applications."
            },
            {
                "role": "user",
                "content": f"Write a Statement of Purpose for a Master's program in {sop_data['field']}."
            },
            {
                "role": "assistant",
                "content": sop_data['text']
            }
        ]
    }


# Format all SOPs
training_examples = []
for sop in dataset:
    example = format_for_training(sop)
    training_examples.append(example)

print(f"✓ Formatted {len(training_examples)} examples for training")

# Show one example (truncated for readability)
print("\nExample formatted data:")
example = training_examples[0].copy()

# Truncate assistant content for display
original_content = example['messages'][2]['content']
example['messages'][2]['content'] = original_content[:100] + "..." if len(original_content) > 100 else original_content

print(json.dumps(example, indent=2, ensure_ascii=False))


✓ Formatted 500 examples for training

Example formatted data:
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert at writing compelling Statements of Purpose for graduate school applications."
    },
    {
      "role": "user",
      "content": "Write a Statement of Purpose for a Master's program in Global Development / Sustainability / Management."
    },
    {
      "role": "assistant",
      "content": "STATEMENT OF PURPOSE:\n\nI was born and raised in the quiet valleys of Khyber Pakhtunkhwa, Pakistan, w..."
    }
  ]
}


## Split Data

Divide dataset into:
- **Training (80%)**: Model learns from these
- **Validation (20%)**: Evaluate performance during training

Using random shuffle with fixed seed for reproducibility.

In [15]:
# Split into train and validation sets
import random

# Set seed for reproducibility
random.seed(42)
random.shuffle(training_examples)

# 80/20 split
split_idx = int(len(training_examples) * 0.8)
train_data = training_examples[:split_idx]
val_data = training_examples[split_idx:]

print(f"{'='*40}")
print(f"Dataset Split:")
print(f"{'='*40}")
print(f"Training examples:   {len(train_data)} (80%)")
print(f"Validation examples: {len(val_data)} (20%)")
print(f"Total:               {len(training_examples)}")
print(f"{'='*40}")

Dataset Split:
Training examples:   400 (80%)
Validation examples: 100 (20%)
Total:               500


## Export to JSONL

Save formatted data to `.jsonl` files (JSON Lines format):
- One JSON object per line
- Standard format for LLM fine-tuning
- Compatible with HuggingFace Trainer

Files will be used in next notebook for training.

In [16]:
def save_to_jsonl(data: List[Dict], filepath: str):
    """
    Save data to JSONL format (one JSON per line).

    Args:
        data: List of training examples
        filepath: Output file path
    """
    with open(filepath, 'w', encoding='utf-8') as f:
        for example in data:
            json_line = json.dumps(example, ensure_ascii=False)
            f.write(json_line + '\n')

# Save train and validation sets
save_to_jsonl(train_data, 'train.jsonl')
save_to_jsonl(val_data, 'val.jsonl')

print("✓ Saved training data to: train.jsonl")
print("✓ Saved validation data to: val.jsonl")

# Verify file sizes
import os
train_size = os.path.getsize('train.jsonl') / (1024 * 1024)  # MB
val_size = os.path.getsize('val.jsonl') / (1024 * 1024)  # MB

print(f"\nFile sizes:")
print(f"  train.jsonl: {train_size:.2f} MB")
print(f"  val.jsonl: {val_size:.2f} MB")

✓ Saved training data to: train.jsonl
✓ Saved validation data to: val.jsonl

File sizes:
  train.jsonl: 1.64 MB
  val.jsonl: 0.41 MB


## Download Prepared Data

Download the processed files to save locally and use in training notebook.

In [20]:
# Download prepared datasets
from google.colab import files

print("Downloading training files...")
files.download('train.jsonl')
files.download('val.jsonl')

print("\n✓ Files downloaded!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Files downloaded!


In [21]:
# Cell 9: Dataset summary
print("="*40)
print("DATA PREPARATION COMPLETE")
print("="*40)
print(f"\n📊 Dataset Statistics:")
print(f"   Total SOPs collected:     500")
print(f"   Training examples:        {len(train_data)}")
print(f"   Validation examples:      {len(val_data)}")
print(f"\n📁 Output Files:")
print(f"   train.jsonl:              {train_size:.2f} MB")
print(f"   val.jsonl:                {val_size:.2f} MB")
print(f"\n✅ Ready for fine-tuning!")
print("="*40)

DATA PREPARATION COMPLETE

📊 Dataset Statistics:
   Total SOPs collected:     500
   Training examples:        400
   Validation examples:      100

📁 Output Files:
   train.jsonl:              1.64 MB
   val.jsonl:                0.41 MB

✅ Ready for fine-tuning!
